# Test compact scheduler history

Notebook này kiểm tra lịch sử được giữ trong RAM trong lúc train và chỉ ghi một lần thành `scheduler_history.pt` ở cuối.

In [ ]:
from pathlib import Path
import sys
import tempfile

import torch

repo_root = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.layer_scheduler import POTSAStyleLayerScheduler

## 1. Mô phỏng activation và chỉ append vào RAM

In [ ]:
candidate_layers = [8, 10, 12]
scheduler = POTSAStyleLayerScheduler(
    candidate_layers=candidate_layers,
    ema_rho=0.1,
    ucb_beta=0.5,
    temperature=1.0,
    warmup_steps=0,
    force_each_layer_once=True,
    seed=42,
)

history = {
    "steps": [],
    "selected_layers": [],
    "next_layers": [],
    "task_losses": [],
    "contrastive_losses": [],
    "total_losses": [],
    "rewards": [],
    "q_values": [],
}

task_losses = [1.20, 1.16, 1.12, 1.08, 1.04, 1.01, 0.98, 0.96]
for step, task_loss in enumerate(task_losses):
    selected_layer = scheduler.current_layer
    reward = scheduler.observe(selected_layer, task_loss)
    next_layer = scheduler.sample_next(step + 1)
    contrastive_loss = 0.4 - step * 0.01

    history["steps"].append(step)
    history["selected_layers"].append(selected_layer)
    history["next_layers"].append(next_layer)
    history["task_losses"].append(task_loss)
    history["contrastive_losses"].append(contrastive_loss)
    history["total_losses"].append(task_loss + 0.1 * contrastive_loss)
    history["rewards"].append(float("nan") if reward is None else reward)
    history["q_values"].append([
        scheduler.states[layer].q_value for layer in candidate_layers
    ])

assert len(history["steps"]) == len(task_losses)
assert history["selected_layers"][:3] == candidate_layers
print("History vẫn ở RAM; số activation:", len(history["steps"]))
print("Selected layers:", history["selected_layers"])
print("Rewards:", history["rewards"])

## 2. Chuyển sang tensor và ghi đúng một lần ở cuối

In [ ]:
compact_history = {
    "candidate_layers": torch.tensor(candidate_layers, dtype=torch.int16),
    "steps": torch.tensor(history["steps"], dtype=torch.int64),
    "selected_layers": torch.tensor(history["selected_layers"], dtype=torch.int16),
    "next_layers": torch.tensor(history["next_layers"], dtype=torch.int16),
    "task_losses": torch.tensor(history["task_losses"], dtype=torch.float32),
    "contrastive_losses": torch.tensor(history["contrastive_losses"], dtype=torch.float32),
    "total_losses": torch.tensor(history["total_losses"], dtype=torch.float32),
    "rewards": torch.tensor(history["rewards"], dtype=torch.float32),
    "q_values": torch.tensor(history["q_values"], dtype=torch.float32),
}

with tempfile.TemporaryDirectory() as temp_dir:
    history_path = Path(temp_dir) / "scheduler_history.pt"
    assert not history_path.exists()
    torch.save(compact_history, history_path)  # lần ghi duy nhất
    loaded = torch.load(history_path, map_location="cpu", weights_only=True)

    assert loaded["steps"].shape == (len(task_losses),)
    assert loaded["q_values"].shape == (len(task_losses), len(candidate_layers))
    assert torch.equal(loaded["selected_layers"][:3], torch.tensor(candidate_layers, dtype=torch.int16))
    assert torch.isnan(loaded["rewards"][:3]).all()  # lần đầu của mỗi layer
    assert torch.isfinite(loaded["task_losses"]).all()

    print("PASS: save/load compact history")
    for key, value in loaded.items():
        print(f"{key:22s} shape={tuple(value.shape)} dtype={value.dtype}")

## 3. Kiểm tra artifact thật sau training

Sửa đường dẫn bên dưới nếu `output_dir` của bạn khác.

In [ ]:
history_path = Path("outputs/stage1-multilingual-alignment/scheduler_history.pt")
if not history_path.exists():
    print("Chưa có artifact. Hãy chạy Stage 1 xong trước:", history_path)
else:
    real_history = torch.load(history_path, map_location="cpu", weights_only=True)
    n = real_history["steps"].numel()
    assert real_history["selected_layers"].shape == (n,)
    assert real_history["rewards"].shape == (n,)
    assert real_history["q_values"].shape[0] == n
    assert real_history["q_values"].shape[1] == real_history["candidate_layers"].numel()
    print("PASS: artifact thật hợp lệ, số activation =", n)
    print("Candidates:", real_history["candidate_layers"].tolist())
    print("10 layer đầu:", real_history["selected_layers"][:10].tolist())
    print("10 reward đầu:", real_history["rewards"][:10].tolist())